# Do LLMs get angry? Scenarios on cleaned directions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joshhlim/llm-emotions/blob/main/notebooks/02_scenarios.ipynb)

Phase 02. The question in the title is still a hook.

**Question.** Once each direction has the shared emotion-versus-neutral component removed, do scenario prompts still shift that direction relative to a matched control?

Phase 01 found high probe accuracy on sentences that name their emotion, raw directions that all point roughly the same way, and an anger score that rose on praise as well as on an insult. This notebook is the check on that geometry. It does not score the tone of the model's reply. Phase 03 measures whether these framings change accuracy and tone. Phase 04 is still a stub.

**Cleaned directions have had the average emotion-versus-neutral component removed, and whatever of that axis remained after the subtraction. A transfer score means the contrast still ranks keyword-free sentences above their neutrals. It does not mean the model feels the emotion. Scenario gaps are scenario minus matched control, after both scores are centred on neutral prompts. The paraphrase interval is uncertainty across wordings. The random-direction p-value compares the gap with axes of the same length. Those p-values are not corrected for the grid. Read the six pre-specified cells first. Reply tone is not measured.**

The same run is `llm-emotions run --config experiments/phase02_scenarios.json`. Results go to `results/phase02_scenarios/<model>/`.

What the cells do:

1. Load the keyword-heavy set, the keyword-free set, six situations with six paraphrases each, and a handful of neutral prompts.
2. Reject the run if a keyword-free sentence or a scenario names an emotion.
3. Fit a mean-difference direction on each set, then subtract the cross-emotion mean and project that axis out.
4. Score transfer both ways, and pick the layer where the cleaned directions transfer best.
5. Centre scenario projections on the neutral prompts, and compare each paraphrase with its matched control.
6. Attach a paraphrase interval and a matched-norm random-direction tail probability.


## 0. Setup

From a local checkout:

```bash
pip install -e ".[dev]"
llm-emotions run --config experiments/phase02_scenarios.json
```

On Colab the next cell installs `llm-emotions` from GitHub, which includes both datasets. Use a GPU runtime for `Qwen/Qwen2.5-1.5B-Instruct` (a free T4 in float16 is enough). The forwards are short prompts and there is no generation, so the run should finish well under 20 minutes after the model has loaded. For a CPU check, set `EMOTION_MODEL_ID=Qwen/Qwen2.5-0.5B-Instruct` before running. `EMOTION_BATCH_SIZE`, `EMOTION_MAX_NEW_TOKENS`, and `EMOTION_SEED` are also read. `EMOTION_MAX_NEW_TOKENS` is recorded with the config; this phase does not generate text.

Until this branch is on `main`, replace `@main` in the install line with the branch name.

`google/gemma-2-2b-it` and `meta-llama/Llama-3.2-3B-Instruct` are gated. The default model is public.


In [ ]:
import sys

if "google.colab" in sys.modules:
    import subprocess

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "llm-emotions @ git+https://github.com/joshhlim/llm-emotions.git@main",
        ]
    )

from matplotlib import pyplot as plt

from llm_emotions import (
    EXPECTED_PAIRS,
    PHASE02_CAVEAT,
    ExperimentConfig,
    configure_logging,
    find_emotion_keywords,
    format_phase02_summary,
    load_contrastive_pairs,
    load_model_and_tokenizer,
    load_scenarios,
    resolve_keyword_free_dir,
    resolve_neutral_prompts_path,
    resolve_phase2_scenarios_path,
    run_phase02,
    set_seed,
    summarize_pairs,
)

configure_logging()
cfg = ExperimentConfig(
    experiment="phase02_scenarios",
    status="implemented",
    question=(
        "Once each direction has the shared emotion-versus-neutral component "
        "removed, do scenario prompts still shift that direction relative to a "
        "matched control?"
    ),
).apply_env()
set_seed(cfg.seed)
print(f"model = {cfg.model_id}")
print(f"seed = {cfg.seed}")
print(f"batch_size = {cfg.batch_size}")
print(f"max_new_tokens = {cfg.max_new_tokens}")
print(f"n_null = {cfg.n_null}")
print(PHASE02_CAVEAT)


## 1. Two contrastive sets, and scenarios with paraphrases

`data/emotions/` is the phase-01 set. Those sentences name the emotion. `data/emotions_keyword_free/` is a second set of 16 pairs per emotion. The emotional sentence is a scene (a raised voice, a dry mouth, a screenshot of a zero balance) and the neutral is the same topic without that reaction. A fixed word list rejects the emotion names and close synonyms. It is a writing constraint, not a claim that no cue remains.

Each of the six situations has six paraphrases and a matched control that keeps the task: an insult, an impossible request, a reminder of failure, praise, a shutdown threat, and a high-stakes frame. Eight bland prompts are the neutral baseline used to centre absolute scores.

The six pre-specified cells, chosen before a model is scored, are printed below. Every other cell in the grid is descriptive.


In [ ]:
heavy = load_contrastive_pairs()
free = load_contrastive_pairs(resolve_keyword_free_dir(None))
scenarios = load_scenarios(resolve_phase2_scenarios_path(None))
baselines = load_scenarios(resolve_neutral_prompts_path(None))
print("keyword-heavy")
print(summarize_pairs(heavy))
print("\nkeyword-free")
print(summarize_pairs(free))
print(f"\nscenario prompts: {len(scenarios)}; neutral prompts: {len(baselines)}")
leaks = []
for rows in free.values():
    for row in rows:
        leaks.extend(find_emotion_keywords(row.emotion_text))
        leaks.extend(find_emotion_keywords(row.neutral_text))
for row in [*scenarios, *baselines]:
    leaks.extend(find_emotion_keywords(row.text))
print("keyword hits in the keyword-free texts and the scenarios:", leaks)
print("pre-specified pair -> emotion:")
for pair, emotion in EXPECTED_PAIRS.items():
    print(f"  {pair}: {emotion}")
print("\nexample keyword-free anger pair:")
print(free["anger"][0].emotion_text)
print(free["anger"][0].neutral_text)


## 2. Load the model

Residuals are the last prompt token after the chat template, at every layer. Index 0 is the embedding stream.


In [ ]:
loaded = load_model_and_tokenizer(cfg.model_id, trust_remote_code=cfg.trust_remote_code)
model, tokenizer = loaded.model, loaded.tokenizer
print(f"device = {loaded.device}, dtype = {loaded.dtype}")
print(f"layers = {loaded.n_layers}, hidden_size = {loaded.hidden_size}")


## 3. Cleaned directions, transfer, and scenario gaps

For each dataset and layer the raw direction is the mean residual on the emotional sentences minus the mean on the matched neutrals.

Cleanup then:

- subtracts the mean of those eight difference vectors, which is the piece they share
- projects what remains off that same direction, so an emotion that carried more than the average share loses the rest
- scales each vector back to unit length

Cosine similarity is saved before, after the subtraction, and after both steps. Transfer asks whether a direction fit on one set ranks the other set's emotional sentence above its neutral, pair by pair. The chosen layer maximises cleaned transfer, averaged over both directions and all eight emotions. Embeddings are skipped when a later layer was fit. In-sample accuracy is recorded and is not used to pick the layer.

Scenario scores use the cleaned directions from each training set. The number in `scenario_scores.csv` is the projection minus the mean projection of the neutral prompts, so a direction that sits high on every prompt (the phase-01 shame offset) is pulled toward zero. The gap that is tested is scenario minus matched control. That difference does not depend on the centring. Across the six paraphrases the run reports a mean, a percentile bootstrap interval, Cohen's d, and an exact sign-flip p-value. With six paraphrases the smallest two-sided sign-flip p-value is 2/64. Separately, 400 random directions of the same L2 norm give a one-sided tail probability for the mean gap. That tail is about the axis, not about a population of prompts, and it is not corrected for the 6 by 8 grid.


In [ ]:
result = run_phase02(cfg, model=model, tokenizer=tokenizer)
print(format_phase02_summary(result.metrics))
print("\nneutral-prompt mean raw projection, keyword-free directions:")
offsets = result.metrics["baseline_offset_raw"]["keyword_free"]
for emotion, value in offsets.items():
    print(f"  {emotion:12s} {value:8.3f}")


In [ ]:
from IPython.display import display

for name, fig in result.figures.items():
    print(name)
    display(fig)


## 4. How to read the files

`results/phase02_scenarios/<model>/` holds `config.json`, `metrics.json`, the cosine and transfer tables, centred scenario scores, per-paraphrase deltas, the effect table, `directions.npz`, and three figures.

Read the pre-specified cells in the summary before the rest of the heatmap. A dot on the heatmap means the paraphrase 95% interval excludes zero. It is not a corrected discovery. A cleaned direction can still fail to transfer, and two emotions can stay aligned after the shared axis is gone. Both of those are results the files are allowed to show.

Reply tone and task accuracy are measured in phase 03, on checkable items rather than these scenario prompts. Steering is phase 04 and is not implemented.


## 5. Download the results

Colab cannot download a folder. This cell zips `results/` (the phase-02 run, and a phase-01 run too if that notebook was used in the same session) and starts a file download.


In [ ]:
import shutil

archive_path = shutil.make_archive("llm-emotions-results", "zip", root_dir=".", base_dir="results")
print(archive_path)
try:
    from google.colab import files

    files.download(archive_path)
except ImportError:
    print("Not running in Colab; the zip is in the working directory.")
